# Rule-Based Market Regimes

Classifies market regimes through price and volatility rules and backtests corresponding exposure.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings("ignore")


class StrategyConfig:
    """
    Central configuration class for all strategy parameters.
    All inputs are configurable here - no hardcoded values in the strategy logic.
    """

    START_DATE = "2019-12-31"
    END_DATE = "2024-12-31"
    REGIME_DETECTION_TICKER = "SPY"
    TRADING_UNIVERSE = ["SPY"]
    BENCHMARK_TICKER = "URTH"
    NUM_REGIMES = 2
    REGIME_LOOKBACK = 252
    MIN_OBSERVATIONS = 100
    POSITION_SIZE_HIGH_VOL = 0.3
    POSITION_SIZE_LOW_VOL = 1.0
    REGIME_PROB_THRESHOLD = 0.7
    ENABLE_STOP_LOSS = False
    STOP_LOSS_PCT = -0.1
    COMMISSION_PCT = 0.001
    SLIPPAGE_PCT = 0.0005
    INITIAL_CAPITAL = 100000


config = StrategyConfig()


class YahooDataFetcher:
    """
    Fetches historical OHLCV data from Yahoo Finance.
    Critical: We fetch daily OHLC data to implement proper execution logic.
    """

    def get_historical_data(self, ticker, start_date, end_date):
        """
        Fetch historical daily data for a ticker.

        Returns DataFrame with: Open, High, Low, Close, Volume
        No forward-filling or back-filling - only real data points.
        """
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data:
                print(f"Warning: No data returned for {ticker}")
                return None
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date")
            df.set_index("date", inplace=True)
            df = df[["open", "high", "low", "close", "volume"]].copy()
            df.columns = ["Open", "High", "Low", "Close", "Volume"]
            df = df.apply(pd.to_numeric, errors="coerce")
            df = df.dropna()
            print(
                f"Downloaded {len(df)} data points for {ticker} from {df.index[0].date()} to {df.index[-1].date()}"
            )
            return df
        except Exception as e:
            print(f"Error fetching data for {ticker}: {str(e)}")
            return None


class RegimeDetector:
    """
    Simplified regime detection based on rolling volatility.

    CRITICAL CHANGES FROM ORIGINAL:
    - Uses expanding/rolling window to avoid look-ahead bias
    - Simpler than Markov Switching for production (more stable)
    - Classifies regimes based on realized volatility
    - All calculations use only T-1 information for decision at T
    """

    def __init__(self, lookback_window=252):
        self.lookback_window = lookback_window

    def calculate_regimes(self, prices_close):
        """
        Calculate market regimes based on rolling volatility.

        Logic:
        - Calculate rolling volatility of returns
        - Compare to historical average
        - High vol regime = volatility > median
        - Low vol regime = volatility <= median

        Returns DataFrame with regime probabilities (0=high vol, 1=low vol)
        """
        returns = prices_close.pct_change()
        rolling_vol = returns.rolling(window=21).std()
        expanding_median_vol = rolling_vol.expanding(min_periods=self.lookback_window).median()
        regime_probs = pd.DataFrame(index=prices_close.index)
        regime_probs["Regime_0_Prob"] = (rolling_vol > expanding_median_vol).astype(float)
        regime_probs["Regime_1_Prob"] = (rolling_vol <= expanding_median_vol).astype(float)
        regime_probs["Most_Likely_Regime"] = (regime_probs["Regime_1_Prob"] > 0.5).astype(int)
        regime_probs["Rolling_Vol"] = rolling_vol
        regime_probs["Expanding_Median_Vol"] = expanding_median_vol
        return regime_probs


class SignalGenerator:
    """
    Generates trading signals based on regime detection.

    CRITICAL: Follows strict T-1 to T logic to avoid look-ahead bias.
    Signal at Close[t] -> Execute at Open[t+1]
    """

    def __init__(self, config):
        self.config = config

    def generate_signals(self, regime_data, prices_df):
        """
        Generate trading signals based on regime probabilities.

        Signal Logic:
        - At Close[t], observe regime probabilities
        - If in Low Vol Regime (1): Signal = config.POSITION_SIZE_LOW_VOL
        - If in High Vol Regime (0): Signal = config.POSITION_SIZE_HIGH_VOL
        - Signal will be executed at Open[t+1]

        Returns: DataFrame with 'Signal' column (target position size)
        """
        signals = pd.DataFrame(index=prices_df.index)
        signals["Regime"] = regime_data["Most_Likely_Regime"]
        signals["Regime_1_Prob"] = regime_data["Regime_1_Prob"]
        signals["Target_Position"] = 0.0
        signals.loc[signals["Regime"] == 1, "Target_Position"] = self.config.POSITION_SIZE_LOW_VOL
        signals.loc[signals["Regime"] == 0, "Target_Position"] = self.config.POSITION_SIZE_HIGH_VOL
        signals["Signal"] = signals["Target_Position"].shift(1)
        signals.iloc[: self.config.MIN_OBSERVATIONS, signals.columns.get_loc("Signal")] = 0.0
        signals["Signal"] = signals["Signal"].ffill()
        signals["Signal"] = signals["Signal"].fillna(0.0)
        return signals


class Backtester:
    """
    Backtesting engine that simulates realistic trading execution.

    CRITICAL FEATURES:
    - Executes trades at Open[t+1] based on signal at Close[t]
    - Values portfolio at Open[t+1], not Close[t+1]
    - Calculates returns as Open[t+2]/Open[t+1] - 1
    - Applies realistic commissions and slippage
    - Implements stop-loss if enabled
    - No look-ahead bias in any calculation
    """

    def __init__(self, config):
        self.config = config

    def run_backtest(self, signals, prices_df):
        """
        Execute backtest with proper trading flow.

        Trading Flow:
        1. At Close[t]: Observe signal, decide target position
        2. At Open[t+1]: Execute trade, value portfolio
        3. Returns calculated as Open[t+2]/Open[t+1] - 1
        """
        bt = pd.DataFrame(index=prices_df.index)
        bt["Open"] = prices_df["Open"]
        bt["Close"] = prices_df["Close"]
        bt["High"] = prices_df["High"]
        bt["Low"] = prices_df["Low"]
        bt["Signal"] = signals["Signal"]
        bt["Position"] = 0.0
        bt["Cash"] = float(self.config.INITIAL_CAPITAL)
        bt["Holdings"] = 0.0
        bt["Portfolio_Value"] = float(self.config.INITIAL_CAPITAL)
        bt["Shares"] = 0.0
        bt["Trade"] = 0.0
        bt["Commission"] = 0.0
        bt["Slippage"] = 0.0
        for i in range(1, len(bt)):
            prev_idx = bt.index[i - 1]
            curr_idx = bt.index[i]
            bt.loc[curr_idx, "Position"] = bt.loc[prev_idx, "Position"]
            bt.loc[curr_idx, "Cash"] = bt.loc[prev_idx, "Cash"]
            bt.loc[curr_idx, "Shares"] = bt.loc[prev_idx, "Shares"]
            target_position = bt.loc[curr_idx, "Signal"]
            current_position = bt.loc[prev_idx, "Position"]
            if abs(target_position - current_position) > 0.001:
                trade = target_position - current_position
                bt.loc[curr_idx, "Trade"] = trade
                execution_price = bt.loc[curr_idx, "Open"]
                portfolio_value = bt.loc[prev_idx, "Portfolio_Value"]
                trade_value = trade * portfolio_value
                total_cost_pct = self.config.COMMISSION_PCT + self.config.SLIPPAGE_PCT
                costs = abs(trade_value) * total_cost_pct
                bt.loc[curr_idx, "Commission"] = abs(trade_value) * self.config.COMMISSION_PCT
                bt.loc[curr_idx, "Slippage"] = abs(trade_value) * self.config.SLIPPAGE_PCT
                if trade > 0:
                    shares_to_buy = trade_value / execution_price
                    bt.loc[curr_idx, "Shares"] = bt.loc[prev_idx, "Shares"] + shares_to_buy
                    bt.loc[curr_idx, "Cash"] = bt.loc[prev_idx, "Cash"] - trade_value - costs
                else:
                    shares_to_sell = -trade_value / execution_price
                    bt.loc[curr_idx, "Shares"] = bt.loc[prev_idx, "Shares"] - shares_to_sell
                    bt.loc[curr_idx, "Cash"] = bt.loc[prev_idx, "Cash"] - trade_value - costs
                bt.loc[curr_idx, "Position"] = target_position
            bt.loc[curr_idx, "Holdings"] = bt.loc[curr_idx, "Shares"] * bt.loc[curr_idx, "Open"]
            bt.loc[curr_idx, "Portfolio_Value"] = (
                bt.loc[curr_idx, "Cash"] + bt.loc[curr_idx, "Holdings"]
            )
            if self.config.ENABLE_STOP_LOSS and bt.loc[curr_idx, "Position"] > 0:
                entry_value = bt.loc[prev_idx, "Portfolio_Value"]
                current_value = bt.loc[curr_idx, "Portfolio_Value"]
                if current_value / entry_value - 1 <= self.config.STOP_LOSS_PCT:
                    bt.loc[curr_idx, "Trade"] = -bt.loc[curr_idx, "Position"]
                    bt.loc[curr_idx, "Position"] = 0.0
                    liquidation_value = bt.loc[curr_idx, "Shares"] * bt.loc[curr_idx, "Open"]
                    costs = liquidation_value * total_cost_pct
                    bt.loc[curr_idx, "Cash"] = liquidation_value - costs
                    bt.loc[curr_idx, "Holdings"] = 0.0
                    bt.loc[curr_idx, "Shares"] = 0.0
                    bt.loc[curr_idx, "Portfolio_Value"] = bt.loc[curr_idx, "Cash"]
        bt["Returns"] = bt["Open"].pct_change()
        bt["Strategy_Returns"] = bt["Portfolio_Value"].pct_change()
        bt["Cumulative_Returns"] = (1 + bt["Strategy_Returns"]).cumprod()
        return bt

    def run_benchmark(self, prices_df):
        """
        Calculate buy-and-hold benchmark returns.
        Uses same execution logic: value at Open, returns = Open[t]/Open[t-1] - 1
        """
        bench = pd.DataFrame(index=prices_df.index)
        bench["Open"] = prices_df["Open"]
        initial_shares = self.config.INITIAL_CAPITAL / bench["Open"].iloc[0]
        bench["Portfolio_Value"] = initial_shares * bench["Open"]
        bench["Returns"] = bench["Portfolio_Value"].pct_change()
        bench["Cumulative_Returns"] = (1 + bench["Returns"]).cumprod()
        return bench


class PerformanceAnalyzer:
    """
    Comprehensive performance metrics calculation.
    All metrics calculated on actual trading returns.
    """

    @staticmethod
    def calculate_metrics(returns_series, benchmark_returns, risk_free_rate=0.02):
        """
        Calculate comprehensive performance metrics.

        Args:
            returns_series: Strategy returns (daily)
            benchmark_returns: Benchmark returns (daily)
            risk_free_rate: Annual risk-free rate (default 2%)
        """
        returns = returns_series.dropna()
        bench_returns = benchmark_returns.dropna()
        ann_factor = 252
        total_return = (1 + returns).prod() - 1
        benchmark_total = (1 + bench_returns).prod() - 1
        n_years = len(returns) / ann_factor
        ann_return = (1 + total_return) ** (1 / n_years) - 1
        bench_ann_return = (1 + benchmark_total) ** (1 / n_years) - 1
        volatility = returns.std() * np.sqrt(ann_factor)
        bench_volatility = bench_returns.std() * np.sqrt(ann_factor)
        excess_return = ann_return - risk_free_rate
        sharpe = excess_return / volatility if volatility > 0 else 0
        bench_excess = bench_ann_return - risk_free_rate
        bench_sharpe = bench_excess / bench_volatility if bench_volatility > 0 else 0
        cum_returns = (1 + returns).cumprod()
        running_max = cum_returns.expanding().max()
        drawdown = (cum_returns - running_max) / running_max
        max_dd = drawdown.min()
        bench_cum = (1 + bench_returns).cumprod()
        bench_running_max = bench_cum.expanding().max()
        bench_dd = (bench_cum - bench_running_max) / bench_running_max
        bench_max_dd = bench_dd.min()
        calmar = ann_return / abs(max_dd) if max_dd != 0 else 0
        bench_calmar = bench_ann_return / abs(bench_max_dd) if bench_max_dd != 0 else 0
        win_rate = (returns > 0).sum() / len(returns)
        wins = returns[returns > 0]
        losses = returns[returns < 0]
        avg_win = wins.mean() if len(wins) > 0 else 0
        avg_loss = losses.mean() if len(losses) > 0 else 0
        downside_returns = returns[returns < 0]
        downside_std = downside_returns.std() * np.sqrt(ann_factor)
        sortino = excess_return / downside_std if downside_std > 0 else 0
        metrics = {
            "Total Return": total_return,
            "Annual Return": ann_return,
            "Volatility": volatility,
            "Sharpe Ratio": sharpe,
            "Sortino Ratio": sortino,
            "Max Drawdown": max_dd,
            "Calmar Ratio": calmar,
            "Win Rate": win_rate,
            "Avg Win": avg_win,
            "Avg Loss": avg_loss,
            "Benchmark Return": benchmark_total,
            "Benchmark Annual Return": bench_ann_return,
            "Benchmark Volatility": bench_volatility,
            "Benchmark Sharpe": bench_sharpe,
            "Benchmark Max DD": bench_max_dd,
            "Benchmark Calmar": bench_calmar,
            "Alpha": ann_return - bench_ann_return,
        }
        return metrics


def plot_results(backtest_df, benchmark_df, signals, metrics):
    """
    Create comprehensive visualization of backtest results.
    All plots use actual data without hardcoded parameters.
    """
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(
        backtest_df.index,
        backtest_df["Cumulative_Returns"],
        label=f"Strategy (Return: {metrics['Total Return']:.2%})",
        linewidth=2,
        color="darkblue",
    )
    ax1.plot(
        benchmark_df.index,
        benchmark_df["Cumulative_Returns"],
        label=f"Benchmark (Return: {metrics['Benchmark Return']:.2%})",
        linewidth=2,
        color="gray",
        alpha=0.7,
    )
    ax1.set_title("Cumulative Returns: Strategy vs Benchmark", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Cumulative Return", fontsize=11)
    ax1.legend(loc="best", fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.fill_between(
        backtest_df.index,
        0,
        backtest_df["Position"],
        alpha=0.3,
        color="green",
        label="Position Size",
    )
    ax2.plot(backtest_df.index, backtest_df["Position"], color="darkgreen", linewidth=1)
    ax2.set_title("Position Sizing Over Time", fontsize=12, fontweight="bold")
    ax2.set_ylabel("Position (%)", fontsize=10)
    ax2.set_ylim([0, 1.1])
    ax2.grid(True, alpha=0.3)
    ax3 = fig.add_subplot(gs[1, 1])
    strategy_cum = backtest_df["Cumulative_Returns"]
    strategy_dd = (strategy_cum - strategy_cum.expanding().max()) / strategy_cum.expanding().max()
    bench_cum = benchmark_df["Cumulative_Returns"]
    bench_dd = (bench_cum - bench_cum.expanding().max()) / bench_cum.expanding().max()
    ax3.fill_between(backtest_df.index, 0, strategy_dd, alpha=0.3, color="red", label="Strategy DD")
    ax3.plot(
        benchmark_df.index, bench_dd, color="gray", linewidth=1.5, alpha=0.7, label="Benchmark DD"
    )
    ax3.set_title("Drawdown Comparison", fontsize=12, fontweight="bold")
    ax3.set_ylabel("Drawdown", fontsize=10)
    ax3.legend(loc="best", fontsize=9)
    ax3.grid(True, alpha=0.3)
    ax4 = fig.add_subplot(gs[2, :])
    ax4.plot(
        signals.index,
        signals["Regime_1_Prob"],
        label="Low Vol Regime Probability",
        color="green",
        linewidth=1.5,
    )
    ax4.axhline(y=0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax4.fill_between(
        signals.index,
        0,
        signals["Regime_1_Prob"],
        where=signals["Regime_1_Prob"] >= 0.5,
        alpha=0.2,
        color="green",
        label="Low Vol Regime",
    )
    ax4.fill_between(
        signals.index,
        0,
        signals["Regime_1_Prob"],
        where=signals["Regime_1_Prob"] < 0.5,
        alpha=0.2,
        color="red",
        label="High Vol Regime",
    )
    ax4.set_title("Market Regime Detection", fontsize=12, fontweight="bold")
    ax4.set_ylabel("Probability", fontsize=10)
    ax4.set_ylim([0, 1])
    ax4.legend(loc="best", fontsize=9)
    ax4.grid(True, alpha=0.3)
    ax5 = fig.add_subplot(gs[3, 0])
    monthly_returns = (
        backtest_df["Strategy_Returns"].resample("ME").apply(lambda x: (1 + x).prod() - 1)
    )
    monthly_returns_pct = monthly_returns * 100
    monthly_returns_pct.index = pd.to_datetime(monthly_returns_pct.index)
    pivot_data = (
        monthly_returns_pct.groupby(
            [monthly_returns_pct.index.year, monthly_returns_pct.index.month]
        )
        .first()
        .unstack()
    )
    sns.heatmap(
        pivot_data,
        annot=True,
        fmt=".1f",
        cmap="RdYlGn",
        center=0,
        cbar_kws={"label": "Return (%)"},
        ax=ax5,
        linewidths=0.5,
    )
    ax5.set_title("Monthly Returns Heatmap (%)", fontsize=12, fontweight="bold")
    ax5.set_xlabel("Month", fontsize=10)
    ax5.set_ylabel("Year", fontsize=10)
    ax6 = fig.add_subplot(gs[3, 1])
    ax6.axis("off")
    metrics_display = [
        ["Metric", "Strategy", "Benchmark"],
        [
            "Annual Return",
            f"{metrics['Annual Return']:.2%}",
            f"{metrics['Benchmark Annual Return']:.2%}",
        ],
        ["Volatility", f"{metrics['Volatility']:.2%}", f"{metrics['Benchmark Volatility']:.2%}"],
        ["Sharpe Ratio", f"{metrics['Sharpe Ratio']:.2f}", f"{metrics['Benchmark Sharpe']:.2f}"],
        ["Max Drawdown", f"{metrics['Max Drawdown']:.2%}", f"{metrics['Benchmark Max DD']:.2%}"],
        ["Calmar Ratio", f"{metrics['Calmar Ratio']:.2f}", f"{metrics['Benchmark Calmar']:.2f}"],
        ["Win Rate", f"{metrics['Win Rate']:.2%}", "-"],
        ["Alpha", f"{metrics['Alpha']:.2%}", "-"],
    ]
    table = ax6.table(
        cellText=metrics_display, cellLoc="center", bbox=[0, 0, 1, 1], colWidths=[0.4, 0.3, 0.3]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    for i in range(3):
        table[0, i].set_facecolor("#40466e")
        table[0, i].set_text_props(weight="bold", color="white")
    plt.suptitle(
        "Regime-Based Trading Strategy - Performance Report",
        fontsize=16,
        fontweight="bold",
        y=0.995,
    )
    return fig


def main():
    """
    Main execution function - orchestrates the entire strategy workflow.
    """
    print("=" * 70)
    print("REGIME-BASED TRADING STRATEGY - PRODUCTION BACKTEST")
    print("=" * 70)
    print(f"\nConfiguration:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Initial Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Commission: {config.COMMISSION_PCT:.2%} per trade")
    print(f"  Slippage: {config.SLIPPAGE_PCT:.2%} per trade")
    print(
        f"  Position Sizes: High Vol={config.POSITION_SIZE_HIGH_VOL:.0%}, Low Vol={config.POSITION_SIZE_LOW_VOL:.0%}"
    )
    print(f"  Stop Loss: {('Enabled' if config.ENABLE_STOP_LOSS else 'Disabled')}")
    print(f"  Benchmark: {config.BENCHMARK_TICKER}")
    print(f"\n{'=' * 70}")
    print("STEP 1: DATA ACQUISITION")
    print("=" * 70)
    fetcher = YahooDataFetcher()
    print(f"\nFetching {config.REGIME_DETECTION_TICKER} for regime detection...")
    regime_data = fetcher.get_historical_data(
        config.REGIME_DETECTION_TICKER, config.START_DATE, config.END_DATE
    )
    if regime_data is None:
        print("ERROR: Could not fetch regime detection data. Exiting.")
        return
    print(f"\nFetching {config.TRADING_UNIVERSE[0]} for trading...")
    trading_data = fetcher.get_historical_data(
        config.TRADING_UNIVERSE[0], config.START_DATE, config.END_DATE
    )
    if trading_data is None:
        print("ERROR: Could not fetch trading data. Exiting.")
        return
    print(f"\nFetching {config.BENCHMARK_TICKER} for benchmark...")
    benchmark_data = fetcher.get_historical_data(
        config.BENCHMARK_TICKER, config.START_DATE, config.END_DATE
    )
    if benchmark_data is None:
        print("ERROR: Could not fetch benchmark data. Exiting.")
        return
    print(f"\n{'=' * 70}")
    print("STEP 2: REGIME DETECTION")
    print("=" * 70)
    detector = RegimeDetector(lookback_window=config.REGIME_LOOKBACK)
    regime_probs = detector.calculate_regimes(regime_data["Close"])
    regime_stats = regime_probs.groupby("Most_Likely_Regime")["Rolling_Vol"].agg(
        ["mean", "std", "count"]
    )
    print("\nRegime Statistics:")
    print(regime_stats)
    print(f"\n{'=' * 70}")
    print("STEP 3: SIGNAL GENERATION")
    print("=" * 70)
    signal_gen = SignalGenerator(config)
    signals = signal_gen.generate_signals(regime_probs, trading_data)
    position_changes = (signals["Signal"].diff() != 0).sum()
    print(f"\nTotal Position Changes: {position_changes}")
    print(f"Average Holding Period: {len(signals) / position_changes:.1f} days")
    print(f"\n{'=' * 70}")
    print("STEP 4: BACKTESTING")
    print("=" * 70)
    backtester = Backtester(config)
    print("\nRunning strategy backtest...")
    backtest_results = backtester.run_backtest(signals, trading_data)
    print("Running benchmark backtest...")
    benchmark_results = backtester.run_benchmark(benchmark_data)
    print(f"\n{'=' * 70}")
    print("STEP 5: PERFORMANCE ANALYSIS")
    print("=" * 70)
    analyzer = PerformanceAnalyzer()
    metrics = analyzer.calculate_metrics(
        backtest_results["Strategy_Returns"], benchmark_results["Returns"]
    )
    print("\n" + "=" * 70)
    print("PERFORMANCE METRICS")
    print("=" * 70)
    print(f"\n{'Metric':<25} {'Strategy':>15} {'Benchmark':>15}")
    print("-" * 70)
    print(
        f"{'Total Return':<25} {metrics['Total Return']:>14.2%} {metrics['Benchmark Return']:>14.2%}"
    )
    print(
        f"{'Annual Return':<25} {metrics['Annual Return']:>14.2%} {metrics['Benchmark Annual Return']:>14.2%}"
    )
    print(
        f"{'Volatility':<25} {metrics['Volatility']:>14.2%} {metrics['Benchmark Volatility']:>14.2%}"
    )
    print(
        f"{'Sharpe Ratio':<25} {metrics['Sharpe Ratio']:>14.2f} {metrics['Benchmark Sharpe']:>14.2f}"
    )
    print(f"{'Sortino Ratio':<25} {metrics['Sortino Ratio']:>14.2f} {'N/A':>15}")
    print(
        f"{'Max Drawdown':<25} {metrics['Max Drawdown']:>14.2%} {metrics['Benchmark Max DD']:>14.2%}"
    )
    print(
        f"{'Calmar Ratio':<25} {metrics['Calmar Ratio']:>14.2f} {metrics['Benchmark Calmar']:>14.2f}"
    )
    print(f"{'Win Rate':<25} {metrics['Win Rate']:>14.2%} {'N/A':>15}")
    print(f"{'Alpha':<25} {metrics['Alpha']:>14.2%} {'N/A':>15}")
    total_commissions = backtest_results["Commission"].sum()
    total_slippage = backtest_results["Slippage"].sum()
    print(f"\n{'Total Commissions':<25} ${total_commissions:>14,.2f}")
    print(f"{'Total Slippage':<25} ${total_slippage:>14,.2f}")
    print(f"{'Total Costs':<25} ${total_commissions + total_slippage:>14,.2f}")
    print(
        f"{'Cost as % of P&L':<25} {(total_commissions + total_slippage) / (backtest_results['Portfolio_Value'].iloc[-1] - config.INITIAL_CAPITAL):>14.2%}"
    )
    print(f"\n{'=' * 70}")
    print("STEP 6: VISUALIZATION")
    print("=" * 70)
    fig = plot_results(backtest_results, benchmark_results, signals, metrics)
    plt.savefig("regime_strategy_performance.png", dpi=300, bbox_inches="tight")
    print("\nPerformance chart saved as 'regime_strategy_performance.png'")
    print(f"\n{'=' * 70}")
    print("EXPORTING RESULTS")
    print("=" * 70)
    export_df = backtest_results[
        [
            "Open",
            "Close",
            "Signal",
            "Position",
            "Trade",
            "Portfolio_Value",
            "Strategy_Returns",
            "Cumulative_Returns",
        ]
    ].copy()
    export_df["Regime"] = signals["Regime"]
    export_df["Benchmark_Returns"] = benchmark_results["Returns"]
    export_df.to_csv("regime_strategy_results.csv")
    print("Results exported to 'regime_strategy_results.csv'")
    metrics_df = pd.DataFrame([metrics]).T
    metrics_df.columns = ["Value"]
    metrics_df.to_csv("regime_strategy_metrics.csv")
    print("Metrics exported to 'regime_strategy_metrics.csv'")
    print(f"\n{'=' * 70}")
    print("BACKTEST COMPLETED SUCCESSFULLY")
    print("=" * 70)
    return (backtest_results, benchmark_results, signals, metrics)


if __name__ == "__main__":
    results = main()
